[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS7150/classdemos/blob/main/optimization/anisotropy.ipynb)

# Anisotropic gradient descent: the zig-zag problem

Consider a quadratic loss that is steep along one coordinate and shallow along another, axis-aligned for simplicity:

$$L(z) = \tfrac12\left(a\,z_x^2 + z_y^2\right), \qquad a \gg 1.$$

Its gradient is

$$\nabla L(z) = (a\,z_x,\; z_y),$$

so the curvature along $x$ (the second derivative $\partial^2 L/\partial z_x^2 = a$) is $a$ times the curvature along $y$ ($\partial^2 L/\partial z_y^2 = 1$). Plain gradient descent with a single shared step size $\eta$ updates both coordinates the same way,

$$z \leftarrow z - \eta \nabla L(z),$$

which, coordinate by coordinate, is $z_x \leftarrow (1-\eta a) z_x$ and $z_y \leftarrow (1-\eta)z_y$. Each coordinate is its own linear recurrence, and it only converges (rather than oscillating or blowing up) when $|1-\eta a| < 1$, i.e.

$$\eta < \frac{2}{a}.$$

That bound is set entirely by the *steep* direction. Any $\eta$ close to it makes $1-\eta a$ close to $-1$, so $z_x$ overshoots the minimum and flips sign on every step — the classic zig-zag. But such an $\eta$ is tiny relative to what the *shallow* direction could tolerate ($\eta<2$ there), so $z_y$ crawls toward zero at the slow rate $(1-\eta)^t$. One shared $\eta$ is stuck compromising between "too big for $x$" and "too small for $y$."

The fix is to give each coordinate its own step size, matched inversely to its own curvature:

$$\eta_x = \frac{\eta}{a}, \qquad \eta_y = \eta.$$

Substituting into the per-coordinate recurrences gives $z_x \leftarrow (1-\eta)z_x$ and $z_y \leftarrow (1-\eta)z_y$: both coordinates now shrink by the *same* factor $(1-\eta)$ every step, so the trajectory heads straight for the minimum instead of zig-zagging. (When the bowl's principal axes are additionally tilted away from the $x$/$y$ axes, as in the live demo below, per-axis rates along $x$/$y$ no longer align perfectly with the bowl's true steep/shallow directions, so they help a great deal but can't fully eliminate the zig-zag — only a rotation back into the bowl's own eigenbasis would.)

See the live version: https://cs7150.github.io/classdemos/demos/optimizers/anisotropy.html

In [ ]:
#@title Setup: anisotropic loss + contour plotting helper (double-click to inspect) { display-mode: "form" }
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# The bowl is L(z) = 1/2 z^T Q z, where Q has eigenvalue `a` (steep) along one
# principal axis and eigenvalue 1 (shallow) along the other, and those axes
# are rotated by `tilt_deg` relative to x/y. This mirrors the live demo's
# computeQ(): a=20, tilt=-25 degrees by default.
def make_Q(a=20.0, tilt_deg=-25.0):
    th = np.deg2rad(tilt_deg)
    c, s = np.cos(th), np.sin(th)
    Q00 = a * s * s + 1.0 * c * c
    Q11 = a * c * c + 1.0 * s * s
    Q01 = (1.0 - a) * s * c
    return np.array([[Q00, Q01], [Q01, Q11]])

def loss(z, Q):
    return 0.5 * z @ Q @ z

def grad(z, Q):
    return Q @ z

def plot_contours_with_paths(Q, paths, labels, colors, title):
    xs = np.linspace(-2, 2, 200)
    ys = np.linspace(-2, 2, 200)
    X, Y = np.meshgrid(xs, ys)
    Z = 0.5 * (Q[0, 0] * X**2 + 2 * Q[0, 1] * X * Y + Q[1, 1] * Y**2)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.contour(X, Y, Z, levels=20, colors='lightgray', linewidths=0.8)
    for path, label, color in zip(paths, labels, colors):
        path = np.array(path)
        ax.plot(path[:, 0], path[:, 1], '-o', color=color, markersize=3, linewidth=1.2, label=label)
    ax.plot(0, 0, 'k*', markersize=12)
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.legend()
    plt.show()

## Plain gradient descent, one shared learning rate

This is the core algorithm: `z <- z - eta * grad(z)`, run for a fixed number
of steps with the same `eta` on both coordinates.

In [ ]:
Q = make_Q(a=20.0, tilt_deg=-25.0)
z0 = np.array([1.2, 0.9])  # start out along the shallow direction, as in the demo

def run_gd(Q, z0, eta, steps=60):
    z = z0.copy()
    path = [z.copy()]
    for _ in range(steps):
        g = grad(z, Q)
        z = z - eta * g
        path.append(z.copy())
    return path

shared_eta = 0.030  # the demo's default shared learning rate
path_shared = run_gd(Q, z0, shared_eta)
print('final loss (shared eta):', loss(path_shared[-1], Q))

## Per-axis learning rates, matched to curvature

Same loop, but each coordinate gets its own `eta`, set roughly inversely
proportional to that axis's curvature (a bigger step where the bowl is
shallower).

In [ ]:
def run_gd_per_axis(Q, z0, eta_vec, steps=60):
    z = z0.copy()
    path = [z.copy()]
    for _ in range(steps):
        g = grad(z, Q)
        z = z - eta_vec * g
        path.append(z.copy())
    return path

eta_vec = np.array([0.090, 0.030])  # x gets the demo's larger, shallow-axis rate
path_per_axis = run_gd_per_axis(Q, z0, eta_vec)
print('final loss (per-axis eta):', loss(path_per_axis[-1], Q))

In [ ]:
plot_contours_with_paths(
    Q,
    [path_shared, path_per_axis],
    ['single shared eta (zig-zags)', 'per-axis eta (goes nearly straight in)'],
    ['tab:orange', 'tab:blue'],
    'Anisotropic gradient descent: shared vs. per-axis learning rate',
)